In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_117_ITO_Delhi_CPCB_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,167.55,245.29,13.29,15.86,19.24,15.57,17.65,1.01,17.95,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-01-02,171.36,243.84,13.88,18.87,21.32,23.65,22.77,1.17,39.83,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-01-03,190.08,246.87,14.08,20.31,22.25,25.45,17.37,1.59,18.77,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-01-04,208.13,281.76,13.83,19.48,21.61,25.87,14.70,1.35,15.90,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-01-05,112.62,193.48,13.67,17.92,20.64,26.46,14.94,1.12,15.90,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,154.11,186.61,23.96,77.55,60.73,48.76,17.42,1.39,14.23,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
362,2024-12-28,93.38,119.11,22.82,68.71,55.09,37.37,17.14,1.24,14.05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
363,2024-12-29,98.52,128.91,18.65,38.12,35.38,32.04,17.84,0.76,14.63,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
364,2024-12-30,94.41,121.45,17.09,32.22,30.94,26.46,13.27,0.96,15.02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 12)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Toluene (µg/m³)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp          0
PM2.5 (µg/m³)      0
PM10 (µg/m³)       0
NO (µg/m³)         0
NO2 (µg/m³)        0
NOx (ppb)          0
NH3 (µg/m³)        0
SO2 (µg/m³)        0
CO (mg/m³)         0
Ozone (µg/m³)      0
Benzene (µg/m³)    0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 11)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         167.55        245.29       13.29        15.86   
1  2024-01-02         171.36        243.84       13.88        18.87   
2  2024-01-03         190.08        246.87       14.08        20.31   
3  2024-01-04         208.13        281.76       13.83        19.48   
4  2024-01-05         112.62        193.48       13.67        17.92   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      19.24        15.57        17.65        1.01          17.95   
1      21.32        23.65        12.77        1.17          39.83   
2      22.25        25.45        17.37        1.59          18.77   
3      21.61        25.87        14.70        1.35          15.90   
4      20.64        26.46        14.94        1.12          15.90   

   Benzene (µg/m³)  
0             3.64  
1             3.55  
2             4.35  
3             2.54  
4             3.78  


In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³)
0,2024-01-01,1.459120,1.402633,-0.791233,-1.405281,-1.361458,-1.125401,1.346196,-0.829551,-0.727353,1.792202
1,2024-01-02,1.529542,1.383593,-0.744251,-1.315229,-1.274115,-0.039354,0.019308,-0.479360,1.135114,1.637193
2,2024-01-03,1.875553,1.423380,-0.728325,-1.272147,-1.235063,0.202587,1.270063,0.439892,-0.657553,3.015049
3,2024-01-04,2.209180,1.881519,-0.748233,-1.296979,-1.261937,0.259040,0.544081,-0.085395,-0.901853,-0.102351
4,2024-01-05,0.443823,0.722319,-0.760974,-1.343651,-1.302669,0.338343,0.609338,-0.588795,-0.901853,2.033326
...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,1.210702,0.632109,0.058417,0.440341,0.380779,0.014411,1.283658,0.002153,-1.044006,-0.119574
362,2024-12-28,0.088201,-0.254230,-0.032361,0.175869,0.143946,1.804775,1.207525,-0.326151,-1.059328,-0.085128
363,2024-12-29,0.183206,-0.125547,-0.364417,-0.739313,-0.683711,1.088361,1.397858,-1.376725,-1.009957,-0.119574
364,2024-12-30,0.107239,-0.223503,-0.488640,-0.915828,-0.870154,0.338343,0.155260,-0.938986,-0.976760,-0.429592


In [10]:
df.to_excel('ITODelhi2024.xlsx', index=False)